# Project Phyto: A Resource-Efficient Edge-AI Framework for Plant Disease Classification
### *Knowledge-Distilled CBAM-Enhanced ShuffleNetV2 with INT8 Quantization & Real-Time Inference Benchmarking*

This Google Colab notebook implements the complete **Phyto** framework pipeline:
1. **Dataset Pipeline**: Real Groundnut Plant Leaf dataset (5 classes: `early_leaf_spot`, `healthy leaf`, `late leaf spot`, `nutrition deficiency`, `rust`).
2. **Teacher Model Training**: High-capacity `ResNet50` fine-tuned for feature extraction reference.
3. **Baseline Model Training**: Standard lightweight `ShuffleNetV2 1.0x` (without Attention or KD).
4. **Proposed Model Training**: `CBAM-ShuffleNetV2 1.0x` (with Channel & Spatial Attention) trained via **Knowledge Distillation** ($L_{KD}$ soft loss + hard loss).
5. **Edge Optimization**: Post-Training **INT8 Quantization** and **ONNX** dynamic axis export.
6. **Comparative Benchmarking**: Accuracy, Macro F1-Score, Confusion Matrices, Latency (ms/image), Throughput (FPS), Model Size (MB), and Parameter counts.

In [ ]:
# Step 1: Environment Setup & Dependencies
!nvidia-smi
!pip install -q torch torchvision torchaudio scikit-learn matplotlib seaborn pandas pillow onnx onnxruntime

In [ ]:
# Step 2: Dataset Extraction & Verification
import os
import zipfile

# Set dataset directory path
DATA_DIR = "Dataset of groundnut plant leaf images for classification and detection/Raw_Data"

# If dataset is zipped in Google Colab, unzip automatically
zip_path = "Dataset of groundnut plant leaf images for classification and detection/Groundnut_Leaf_dataset.zip"
if os.path.exists(zip_path) and not os.path.exists(DATA_DIR):
    print("Unzipping Groundnut Leaf Dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("Dataset of groundnut plant leaf images for classification and detection/")
    print("Unzip complete!")

# List classes and count images
if os.path.exists(DATA_DIR):
    classes = sorted(os.listdir(DATA_DIR))
    print(f"Found {len(classes)} classes:")
    total_imgs = 0
    for c in classes:
        c_path = os.path.join(DATA_DIR, c)
        if os.path.isdir(c_path):
            cnt = len(os.listdir(c_path))
            total_imgs += cnt
            print(f"  - {c}: {cnt} images")
    print(f"Total real images: {total_imgs}")
else:
    print(f"Dataset folder not found at {DATA_DIR}. Please check upload path.")

In [ ]:
# Step 3: Phyto Architecture & Module Imports
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import time
import numpy as np

# Set seeds
torch.manual_seed(42)
np.random.seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Executing on Device: {DEVICE}")

In [ ]:
# Step 4: Dataset Engine & Stratified Splitting (70% Train, 15% Val, 15% Test)
class GroundnutDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]

# Image Transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load files
subdirs = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
class_to_idx = {cls_name: i for i, cls_name in enumerate(subdirs)}
idx_to_class = {i: cls_name for i, cls_name in enumerate(subdirs)}

image_paths, labels = [], []
for cls_name in subdirs:
    cls_dir = os.path.join(DATA_DIR, cls_name)
    for file in os.listdir(cls_dir):
        if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')):
            image_paths.append(os.path.join(cls_dir, file))
            labels.append(class_to_idx[cls_name])

train_p, temp_p, train_l, temp_l = train_test_split(image_paths, labels, test_size=0.30, stratify=labels, random_state=42)
val_p, test_p, val_l, test_l = train_test_split(temp_p, temp_l, test_size=0.50, stratify=temp_l, random_state=42)

train_ds = GroundnutDataset(train_p, train_l, train_transform)
val_ds = GroundnutDataset(val_p, val_l, eval_transform)
test_ds = GroundnutDataset(test_p, test_l, eval_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

print(f"DataLoaders Built! Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

In [ ]:
# Step 5: Define CBAM Attention & Model Architectures
class CBAM(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        reduced = max(in_channels // reduction, 8)
        self.ca_avg = nn.AdaptiveAvgPool2d(1)
        self.ca_max = nn.AdaptiveMaxPool2d(1)
        self.ca_mlp = nn.Sequential(
            nn.Conv2d(in_channels, reduced, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(reduced, in_channels, 1, bias=False)
        )
        self.ca_sig = nn.Sigmoid()
        self.sa_conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
        self.sa_sig = nn.Sigmoid()

    def forward(self, x):
        # Channel Attention
        ca = self.ca_sig(self.ca_mlp(self.ca_avg(x)) + self.ca_mlp(self.ca_max(x)))
        x = x * ca
        # Spatial Attention
        sa_avg = torch.mean(x, dim=1, keepdim=True)
        sa_max, _ = torch.max(x, dim=1, keepdim=True)
        sa = self.sa_sig(self.sa_conv(torch.cat([sa_avg, sa_max], dim=1)))
        x = x * sa
        return x

# 1. Baseline ShuffleNetV2
class BaselineShuffleNetV2(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        base = models.shufflenet_v2_x1_0(weights=models.ShuffleNet_V2_X1_0_Weights.DEFAULT)
        self.features = nn.Sequential(base.conv1, base.maxpool, base.stage2, base.stage3, base.stage4, base.conv5)
        self.fc = nn.Linear(base.fc.in_features, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.mean([2, 3])
        return self.fc(x)

# 2. Proposed CBAM-ShuffleNetV2
class CBAMShuffleNetV2(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        base = models.shufflenet_v2_x1_0(weights=models.ShuffleNet_V2_X1_0_Weights.DEFAULT)
        self.conv1 = base.conv1
        self.maxpool = base.maxpool
        self.stage2 = base.stage2
        self.cbam2 = CBAM(116)
        self.stage3 = base.stage3
        self.cbam3 = CBAM(232)
        self.stage4 = base.stage4
        self.cbam4 = CBAM(464)
        self.conv5 = base.conv5
        self.fc = nn.Sequential(nn.Dropout(0.2), nn.Linear(base.fc.in_features, num_classes))

    def forward(self, x):
        x = self.conv1(x)
        x = self.maxpool(x)
        x = self.cbam2(self.stage2(x))
        x = self.cbam3(self.stage3(x))
        x = self.cbam4(self.stage4(x))
        x = self.conv5(x)
        x = x.mean([2, 3])
        return self.fc(x)

# 3. Teacher ResNet50
class TeacherResNet50(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        base = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        in_f = base.fc.in_features
        base.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_f, num_classes))
        self.model = base

    def forward(self, x):
        return self.model(x)

print("Model architectures initialized successfully!")

In [ ]:
# Step 6: Trainer Engine Functions
def train_model(model, train_loader, val_loader, epochs=15, lr=1e-3, teacher=None, name="model"):
    model = model.to(DEVICE)
    if teacher:
        teacher = teacher.to(DEVICE)
        teacher.eval()
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    ce_loss = nn.CrossEntropyLoss()
    kl_loss = nn.KLDivLoss(reduction="batchmean")
    
    best_acc = 0.0
    history = {"val_loss": [], "val_acc": []}
    T, alpha = 4.0, 0.7
    
    for epoch in range(1, epochs + 1):
        model.train()
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad()
            
            if teacher:
                with torch.no_grad():
                    t_logits = teacher(inputs)
                s_logits = model(inputs)
                loss_ce = ce_loss(s_logits, targets)
                soft_s = F.log_softmax(s_logits / T, dim=1)
                soft_t = F.softmax(t_logits / T, dim=1)
                loss_kd = kl_loss(soft_s, soft_t) * (T ** 2)
                loss = (1.0 - alpha) * loss_ce + alpha * loss_kd
            else:
                outputs = model(inputs)
                loss = ce_loss(outputs, targets)
                
            loss.backward()
            optimizer.step()
            
        scheduler.step()
        
        # Val loop
        model.eval()
        v_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                outputs = model(inputs)
                v_loss += ce_loss(outputs, targets).item() * inputs.size(0)
                preds = outputs.argmax(dim=1)
                correct += preds.eq(targets).sum().item()
                total += targets.size(0)
                
        val_acc = 100.0 * correct / total
        history["val_loss"].append(v_loss / total)
        history["val_acc"].append(val_acc)
        print(f"[{name}] Epoch {epoch:02d}/{epochs:02d} | Val Loss: {v_loss/total:.4f} | Val Acc: {val_acc:.2f}%")
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f"{name}_best.pth")
            
    return history

In [ ]:
# Step 7: Execution Phase 1 - Train Teacher Model (ResNet50)
teacher_model = TeacherResNet50(num_classes=5)
print("Training Teacher Model...")
train_model(teacher_model, train_loader, val_loader, epochs=12, name="teacher_resnet50")

In [ ]:
# Step 8: Execution Phase 2 - Train Baseline Model (ShuffleNetV2)
baseline_model = BaselineShuffleNetV2(num_classes=5)
print("Training Baseline Model...")
train_model(baseline_model, train_loader, val_loader, epochs=15, name="baseline_shufflenetv2")

In [ ]:
# Step 9: Execution Phase 3 - Train Proposed Model (CBAM-ShuffleNetV2 + KD)
proposed_model = CBAMShuffleNetV2(num_classes=5)
print("Training Proposed Model with Knowledge Distillation...")
train_model(proposed_model, train_loader, val_loader, epochs=15, teacher=teacher_model, name="proposed_cbam_kd")

In [ ]:
# Step 10: INT8 Post-Training Quantization
proposed_model.eval().to("cpu")
quantized_model = torch.ao.quantization.quantize_dynamic(
    proposed_model, {nn.Linear, nn.Conv2d}, dtype=torch.qint8
)
torch.save(quantized_model.state_dict(), "proposed_int8.pth")
print("INT8 Quantized Model Saved!")

In [ ]:
# Step 11: Comprehensive Test Benchmarking & Results Comparison
def evaluate_full(model, loader, name, dev=DEVICE):
    model.eval().to(dev)
    all_p, all_t = [], []
    start = time.time()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(dev)
            preds = model(x).argmax(dim=1).cpu().numpy()
            all_p.extend(preds)
            all_t.extend(y.numpy())
            
    tot_t = time.time() - start
    acc = accuracy_score(all_t, all_p) * 100.0
    prec, rec, f1, _ = precision_recall_fscore_support(all_t, all_p, average="macro", zero_division=0)
    cm = confusion_matrix(all_t, all_p)
    
    # Latency benchmarking on 1 item
    dummy = torch.randn(1, 3, 224, 224).to(dev)
    t0 = time.time()
    with torch.no_grad():
        for _ in range(50):
            _ = model(dummy)
            if dev == "cuda": torch.cuda.synchronize()
    lat_ms = ((time.time() - t0) / 50.0) * 1000.0
    
    # Size
    params_m = sum(p.numel() for p in model.parameters()) / 1e6
    
    return {
        "Model Name": name,
        "Accuracy (%)": f"{acc:.2f}%",
        "Macro F1 (%)": f"{f1*100:.2f}%",
        "Latency (ms)": f"{lat_ms:.2f} ms",
        "FPS": f"{1000.0/lat_ms:.1f}",
        "Params (M)": f"{params_m:.2f}M",
        "cm": cm
    }

r_t = evaluate_full(teacher_model, test_loader, "Teacher (ResNet50)")
r_b = evaluate_full(baseline_model, test_loader, "Baseline (ShuffleNetV2)")
r_p = evaluate_full(proposed_model, test_loader, "Proposed (CBAM-ShuffleNet+KD)")
r_q = evaluate_full(quantized_model, test_loader, "Proposed INT8 (Quantized)", dev="cpu")

results_df = pd.DataFrame([r_t, r_b, r_p, r_q]).drop(columns=["cm"])
print("\n=================== COMPARATIVE BENCHMARKING SUMMARY ===================")
display(results_df)

# Confusion Matrix Plot
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for i, r in enumerate([r_t, r_b, r_p, r_q]):
    sns.heatmap(r["cm"], annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=subdirs, yticklabels=subdirs, ax=axes[i])
    axes[i].set_title(r["Model Name"], fontweight="bold")
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("True")
plt.tight_layout()
plt.show()